# 赛道9 Baseline 快速运行 Notebook

本 Notebook 用于一键运行 `sklearn_baseline.py`，生成电力市场储能电站的充放电策略。

每次运行结果具有随机性，不会输出相同的结果。

## 1. 检查并安装依赖

In [ ]:
import subprocess
import sys

required = ['pandas', 'numpy', 'scikit-learn']
missing = []
for pkg in required:
    try:
        __import__(pkg.replace('scikit-learn', 'sklearn'))
        print(f'✅ {pkg} 已安装')
    except ImportError:
        missing.append(pkg)

if missing:
    print(f'正在安装缺失依赖: {missing}')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', *missing])
else:
    print('所有依赖已就绪！')

## 2. 运行 Baseline 脚本

运行 `sklearn_baseline.py`，输出预测电价与充放电策略。

In [ ]:
import os

# 确保在脚本所在目录运行
script_path = './sklearn_baseline.py'
if not os.path.exists(script_path):
    raise FileNotFoundError(f'找不到脚本: {script_path}')

# 执行脚本
exec(open(script_path, encoding='utf-8').read())

## 3. 查看输出结果

生成的文件：
- `output/sklearn_baseline_output.csv` — 预测的节点电价
- `output/output.csv` — 最终的充放电策略（提交文件）

In [ ]:
import pandas as pd

# 查看预测电价
price_df = pd.read_csv('output/sklearn_baseline_output.csv')
print('预测电价预览：')
display(price_df.head(10))
print(f'形状: {price_df.shape}')

In [ ]:
# 查看充放电策略
power_df = pd.read_csv('output/output.csv')
print('充放电策略预览：')
display(power_df.head(10))
print(f'形状: {power_df.shape}')

# 统计充放电情况
power_nonzero = power_df[power_df['power'] != 0]
print(f'\n非零功率记录数: {len(power_nonzero)}')
print(f'充电记录数: {len(power_df[power_df["power"] < 0])}')
print(f'放电记录数: {len(power_df[power_df["power"] > 0])}')

In [ ]:
# 可视化某一天的充放电策略与电价
import matplotlib.pyplot as plt

power_df['times'] = pd.to_datetime(power_df['times'])
power_df['date'] = power_df['times'].dt.date

# 取第一天展示
first_date = power_df['date'].unique()[0]
day_df = power_df[power_df['date'] == first_date].copy()

fig, ax1 = plt.subplots(figsize=(14, 5))

color = 'tab:blue'
ax1.set_xlabel('Time')
ax1.set_ylabel('Price', color=color)
ax1.plot(day_df['times'], day_df['实时价格'], color=color, label='Price')
ax1.tick_params(axis='y', labelcolor=color)

ax2 = ax1.twinx()
color = 'tab:red'
ax2.set_ylabel('Power', color=color)
ax2.step(day_df['times'], day_df['power'], where='post', color=color, label='Power')
ax2.tick_params(axis='y', labelcolor=color)

plt.title(f'Charge/Discharge Strategy & Price on {first_date}')
fig.tight_layout()
plt.show()